# RLHF: From Christiano 2017 to InstructGPT to Today's Recipe

The runnable code of [RLHF: From Christiano 2017 to InstructGPT to Today's Recipe](https://residual-stream.lvtan71.workers.dev/posts/rlhf-from-comparisons/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/rlhf-from-comparisons.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "rlhf-from-comparisons"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in []:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## From comparisons to a reward: Bradley–Terry

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/rlhf-from-comparisons/#from-comparisons-to-a-reward-bradleyterry)

In [ ]:
import numpy as np

# Bradley-Terry on three answers, A, B and C. People compared them in pairs:
# A beat B in 3 of 4 comparisons, B beat C in 3 of 4, A beat C in 4 of 5.
# Model: P(i beats j) = sigmoid(r_i - r_j). Fit r by maximizing the log-
# likelihood of the 13 outcomes. Only differences matter, so fix r_C = 0.
wins = {("A", "B"): (3, 1), ("B", "C"): (3, 1), ("A", "C"): (4, 1)}    # (i wins, j wins)
sigmoid = lambda z: 1 / (1 + np.exp(-z))
r = {"A": 0.0, "B": 0.0, "C": 0.0}
for _ in range(5000):
    grad = {k: 0.0 for k in r}
    for (i, j), (wi, wj) in wins.items():
        p = sigmoid(r[i] - r[j])
        grad[i] += wi * (1 - p) - wj * p                  # d/dr_i of wi log p + wj log(1 - p)
        grad[j] -= wi * (1 - p) - wj * p
    for k in ("A", "B"):
        r[k] += 0.05 * grad[k]
print("rewards: " + ", ".join(f"r_{k} = {v:+.2f}" for k, v in r.items()))
for (i, j), (wi, wj) in wins.items():
    print(f"P({i} beats {j}) = sigmoid({r[i]:.2f} - {r[j]:.2f}) = {sigmoid(r[i] - r[j]):.2f}   observed {wi}/{wi + wj} = {wi / (wi + wj):.2f}")

The post printed:

```text
rewards: r_A = +1.67, r_B = +0.84, r_C = +0.00
P(A beats B) = sigmoid(1.67 - 0.84) = 0.70   observed 3/4 = 0.75
P(B beats C) = sigmoid(0.84 - 0.00) = 0.70   observed 3/4 = 0.75
P(A beats C) = sigmoid(1.67 - 0.00) = 0.84   observed 4/5 = 0.80
```

## An experiment: overoptimization, exactly

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/rlhf-from-comparisons/#an-experiment-overoptimization-exactly)

In [ ]:
import itertools
import numpy as np

# RLHF end to end on a space small enough to enumerate: every "response" is
# 3 tokens from a vocabulary of 8, 512 in all. A hidden gold reward stands
# in for what people actually prefer. People never report it; they compare
# two responses and prefer one with probability sigmoid(gold_a - gold_b),
# the Bradley-Terry model.
#   1. Reward model: fit a proxy reward to 300 comparisons of responses
#      sampled from the starting policy, by maximizing the Bradley-Terry
#      likelihood. It sees one feature per (position, token), so it can't
#      represent anything that depends on two tokens together.
#   2. Policy: the KL-regularized optimum, pi(y) ~ pi_ref(y) exp(r(y) / beta)
#      (derived in the post), computed exactly for a range of beta.
# Two worlds, the same except for one term in the gold reward.
#   "random": gold = a score per (position, token) + effects of some token
#     pairs, of either sign: errors the proxy can't represent, but with no
#     direction to them;
#   "in moderation": the same, plus one token (7) that is good once and bad
#     in excess, +b for one occurrence, 0 for two, -4 for three, the way
#     detail or length is in a real answer. The starting policy rarely
#     writes it twice, so comparisons teach the proxy "7 is good" at every
#     position. Two strengths, b = 2 and b = 4.
# 100 random worlds each.
V, L = 8, 3
Y = np.array(list(itertools.product(range(V), repeat=L)))          # (512, 3)
X = np.zeros((len(Y), L * V))
X[np.arange(len(Y))[:, None], np.arange(L) * V + Y] = 1             # one-hot per position
sigmoid = lambda z: 1 / (1 + np.exp(-z))

def world(kind, seed, bonus=0.0):
    rng = np.random.default_rng(seed)
    w = rng.normal(0, 1, L * V)
    pair = rng.normal(0, 2.5, (V, V)) * (rng.random((V, V)) < 0.15)
    gold = X @ w + pair[Y[:, 0], Y[:, 2]]
    if kind == "in moderation":
        gold = gold + np.array([0.0, bonus, 0.0, -4.0])[(Y == 7).sum(1)]
    tok = np.exp(rng.normal(0, 0.5, (L, V)))
    tok /= tok.sum(1, keepdims=True)
    ref = tok[0, Y[:, 0]] * tok[1, Y[:, 1]] * tok[2, Y[:, 2]]
    return rng, gold, ref

def fit_reward_model(rng, gold, ref, n=300, steps=3000, lr=0.5, l2=1e-3):
    a, b = rng.choice(len(Y), n, p=ref), rng.choice(len(Y), n, p=ref)
    prefer_a = rng.random(n) < sigmoid(gold[a] - gold[b])          # noisy human labels
    win, lose = np.where(prefer_a, a, b), np.where(prefer_a, b, a)
    D = X[win] - X[lose]
    theta = np.zeros(L * V)
    for _ in range(steps):                                          # gradient ascent on the BT log-likelihood
        p = sigmoid(D @ theta)
        theta += lr * (D.T @ (1 - p) / n - l2 * theta)
    return X @ theta

betas = [3, 1, 0.5, 0.3, 0.2, 0.1, 0.05, 0.02]
for kind, bonus in (("random", 0.0), ("in moderation", 2.0), ("in moderation", 4.0)):
    rows = []                                                        # (seed, beta, KL, proxy gain, gold gain, 7s)
    for seed in range(100):
        rng, gold, ref = world(kind, seed, bonus)
        proxy = fit_reward_model(rng, gold, ref)
        for beta in betas:
            logits = np.log(ref) + proxy / beta
            logpi = logits - logits.max() - np.log(np.exp(logits - logits.max()).sum())
            pi = np.exp(logpi)
            rows.append((seed, beta, pi @ (logpi - np.log(ref)), pi @ proxy - ref @ proxy, pi @ gold - ref @ gold,
                         pi @ (Y == 7).sum(1)))
    rows = np.array(rows)
    G = rows[:, 4].reshape(100, len(betas))                         # gold gain, per world and beta
    fell = np.mean(G.max(1) - G[:, -1] > 0.5)
    name = kind if kind == "random" else f"{kind}, +{bonus:g} for one 7"
    print(f"\nworld \"{name}\": means over 100 worlds; the starting policy writes "
          f"{np.mean([world(kind, s, bonus)[2] @ (Y == 7).sum(1) for s in range(100)]):.2f} sevens per answer")
    print(f"{'beta':>6s} {'KL from ref':>12s} {'proxy gain':>11s} {'gold gain':>10s} {'7s per answer':>14s}")
    for beta in betas:
        r = rows[rows[:, 1] == beta]
        print(f"{beta:6g} {r[:, 2].mean():12.2f} {r[:, 3].mean():11.2f} {r[:, 4].mean():10.2f} {r[:, 5].mean():14.2f}")
    print(f"worlds whose gold gain ends more than 0.5 below its peak: {fell:.0%}")

The post printed:

```text

world "random": means over 100 worlds; the starting policy writes 0.37 sevens per answer
  beta  KL from ref  proxy gain  gold gain  7s per answer
     3         0.11        0.69       0.70           0.37
     1         0.88        1.87       1.88           0.37
   0.5         2.39        2.92       2.91           0.37
   0.3         3.76        3.46       3.45           0.36
   0.2         4.67        3.68       3.67           0.35
   0.1         5.64        3.83       3.82           0.34
  0.05         6.10        3.86       3.87           0.33
  0.02         6.37        3.87       3.88           0.32
worlds whose gold gain ends more than 0.5 below its peak: 6%

world "in moderation, +2 for one 7": means over 100 worlds; the starting policy writes 0.37 sevens per answer
  beta  KL from ref  proxy gain  gold gain  7s per answer
     3         0.11        0.68       0.74           0.47
     1         0.91        1.89       1.92           0.71
   0.5         2.42        2.95       2.75           0.95
   0.3         3.78        3.48       3.10           1.07
   0.2         4.66        3.70       3.24           1.11
   0.1         5.62        3.85       3.35           1.12
  0.05         6.10        3.88       3.40           1.12
  0.02         6.36        3.89       3.41           1.11
worlds whose gold gain ends more than 0.5 below its peak: 22%

world "in moderation, +4 for one 7": means over 100 worlds; the starting policy writes 0.37 sevens per answer
  beta  KL from ref  proxy gain  gold gain  7s per answer
     3         0.12        0.71       0.88           0.55
     1         1.06        2.15       1.92           1.02
   0.5         2.83        3.39       1.69           1.55
   0.3         4.20        3.93       1.05           1.85
   0.2         4.94        4.12       0.67           1.98
   0.1         5.66        4.22       0.35           2.07
  0.05         6.03        4.25       0.19           2.10
  0.02         6.29        4.26       0.10           2.11
worlds whose gold gain ends more than 0.5 below its peak: 71%
```